In [ ]:
# Import required libraries
import xml.etree.ElementTree as ET # Parse XML files
import pandas as pd
import re
from collections import Counter

In [ ]:
# Load and parse the MedlinePlus XML file - get first look
tree = ET.parse('mplus_topics_2026-07-07.xml')
root = tree.getroot()

In [6]:
print("root tag:", root.tag)
print("root attributes:", root.attrib)

root tag: health-topics
root attributes: {'total': '2033', 'date-generated': '07/07/2026 02:30:38'}


root has a 'total' attribute, telling us how many topics are supposed to be in there


In [7]:
topics = root.findall('health-topic')
print("number of health-topic elements found:", len(topics))


number of health-topic elements found: 2033


correct parse as it matches root.attrib['total']

In [10]:
#Explore one record
first_topic = topics[0]
print("first topic raw attributes")
print(first_topic.attrib)


first topic raw attributes
{'meta-desc': 'If you are being tested for Type 2 diabetes, your doctor gives you an A1C test. The test is also used to monitor your A1C levels.', 'title': 'A1C', 'url': 'https://medlineplus.gov/a1c.html', 'id': '6308', 'language': 'English', 'date-created': '12/22/2015'}


In [11]:
print("child tags in first topic")
for child in first_topic:
    print(child.tag, "->", (child.text[:80] + "...") if child.text else None)


child tags in first topic
also-called -> Glycohemoglobin...
also-called -> HbA1C...
also-called -> Hemoglobin A1C test...
full-summary -> <p>A1C is a blood test for <a href="https://medlineplus.gov/diabetestype2.html">...
group -> Diagnostic Tests...
group -> Diabetes Mellitus...
language-mapped-topic -> Prueba de hemoglobina glicosilada (HbA1c)...
mesh-heading -> 
...
other-language -> Spanish...
primary-institute -> National Institute of Diabetes and Digestive and Kidney Diseases...
see-reference -> Hemoglobin A1c...
site -> 
...
site -> 
...
site -> 
...
site -> 
...
site -> 
...
site -> 
...
site -> 
...
site -> 
...


From this first record, we can see the general structure of the XML.
Topics have attributes like title, id, language, and meta-desc, plus several child fields. Some fields can appear multiple times, such as group and also-called, so we should keep those as lists when extracting.

In [13]:
#pull out relevant fields( might update later)
records = []

for topic in topics:
    title = topic.get('title')
    lang = topic.get('language')
    topic_id = topic.get('id')
    meta_desc = topic.get('meta-desc')

    summary_el = topic.find('full-summary')
    #some topics might not have so check before pulling .text
    if summary_el is not None and summary_el.text:
        summary_text = summary_el.text
    else:
        summary_text = "" #keep empty rather than drop
    
    #check if html tags are in text
    contains_html = bool(re.search(r'<[a-zA-Z/][^>]*>', summary_text))
    #word count but strip tags first, so we're not counting <p> as a word

    plain_text = re.sub(r'<[^>]+>', ' ', summary_text)
    word_count = len(plain_text.split())

    groups = [g.text for g in topic.findall('group')]

    records.append({
        'id': topic_id,
        'title': title,
        'language': lang,
        'has_meta_desc': meta_desc is not None,
        'word_count': word_count,
        'contains_html': contains_html,
        'group_list': groups,
    })

    



In [14]:
#convert records to a data frame 
df = pd.DataFrame(records)
print("\ndataframe shape:", df.shape)
df.head()


dataframe shape: (2033, 7)


,id,title,language,has_meta_desc,word_count,contains_html,group_list
0,6308,A1C,English,True,197,True,"[Diagnostic Tests, Diabetes Mellitus]"
1,3061,Abdominal Pain,English,True,136,True,"[Digestive System, Symptoms]"
2,122,Abortion,English,True,89,True,"[Pregnancy and Reproduction, Female Reproducti..."
3,2238,Aborto,Spanish,True,87,True,"[Embarazo y reproducción, Sistema reproductor ..."
4,3064,Abscesos,Spanish,True,130,True,[Infecciones]


Below are sanity checks for data quality


In [17]:
#missing titles
print( df['title'].isna().sum())

0


In [19]:
#missing ids
print( df['id'].isna().sum())

#no missing title or ids


0


In [20]:
#check for duplicate titles 
dupe_titles = df[df.duplicated('title', keep=False)].sort_values('title')
print("\nnumber of duplicated titles:", df['title'].duplicated().sum())
print(dupe_titles[['title', 'language']].head(10))


number of duplicated titles: 49
          title language
39      Alcohol  English
40      Alcohol  Spanish
67       Anemia  English
68       Anemia  Spanish
159  Barotrauma  English
160  Barotrauma  Spanish
204       Botox  English
205       Botox  Spanish
227    Bursitis  Spanish
228    Bursitis  English


These titles are english/spanish pairs.

In [21]:
# language split
print("\nlanguage counts:")
print(df['language'].value_counts())


language counts:
language
English    1017
Spanish    1016
Name: count, dtype: int64


Half english/ half spanish pairs. We probably only need. english pairs.but will leave fo noe. 


In [22]:
#check word count distribution 
print(df['word_count'].describe())

count    2033.000000
mean      359.541072
std       288.757642
min         0.000000
25%       145.000000
50%       204.000000
75%       569.000000
max      1295.000000
Name: word_count, dtype: float64


Anything with 0 words is suspicous and needs to be checked. Either a broken record or an empty topic page

In [23]:
empty_topics = df[df['word_count'] == 0]

In [24]:
print("\ntopics with zero words in summary:")
print(empty_topics[['title', 'language']])


topics with zero words in summary:
           title language
1737  Stillbirth  English


In [26]:
#Check long tail - very long topics might need to be split later on 
long_topics = df[df['word_count'] > 1000].sort_values('word_count', ascending=False)

In [27]:
print("\nnumber of topics over 1000 words:", len(long_topics))


number of topics over 1000 words: 73


In [28]:
print(long_topics[['title', 'word_count']].head(10))


                                     title  word_count
1388                         Paro cardíaco        1295
781                         Fallo cardíaco        1231
1012                          Hígado graso        1210
691   Enfermedades del corazón en la mujer        1198
632      Enfermedad de células falciformes        1193
348                  Colesterol malo (LDL)        1185
345                             Colesterol        1179
1526     Pruebas para la salud del corazón        1168
1151            Lesión cerebral traumática        1164
439                         Cáncer de seno        1163


In [ ]:
#HTML TAGS CHECK
# to see the propotion of the data needs cleaning before building an entire pipeline to clean 

In [ ]:
html_count = df['contains_html'].sum()
print(html_count)
print(len(df))


2032
2033


2032 of 2033 topics contain html tags so they need to be cleaned. so the data is useful for the generator

In [ ]:
#Group breakdown 
#Look at groups per topic - a topic can belong to more thna one group

In [35]:
# Only keep English topics for the category analysis
english_df = df[df['language'] == 'English']

all_groups = []

for group_list in english_df['group_list']:
    for g in group_list:
        if g:  # skip empty/None entries just in case
            all_groups.append(g)

In [38]:
group_counts = Counter(all_groups)
print("\ntop 20 groups/categories by number of topics:")
for group, count in group_counts.most_common(20):
    print(f"  {group}: {count}")


top 20 groups/categories by number of topics:
  Infections: 126
  Brain and Nerves: 102
  Blood, Heart and Circulation: 93
  Digestive System: 79
  Bones, Joints and Muscles: 78
  Children and Teenagers: 77
  Female Reproductive System: 69
  Cancers: 66
  Skin, Hair and Nails: 63
  Women: 60
  Immune System: 58
  Food and Nutrition: 57
  Injuries and Wounds: 56
  Genetics/Birth Defects: 56
  Lungs and Breathing: 51
  Endocrine System: 51
  Older Adults: 43
  Mental Health and Behavior: 43
  Pregnancy and Reproduction: 40
  Symptoms: 38


In [ ]:
# Save to CSV to explore the dats before conversion to JSON. 
# Caeful about group_list, use ',' to seperate each string. 


In [39]:
df_to_save = df.copy()
df_to_save['group_list'] = df_to_save['group_list'].apply(lambda g: ', '.join([x for x in g if x]))

In [40]:
df_to_save.to_csv('medlineplus_eda_summary.csv', index=False)
print("\nsaved eda table to medlineplus_eda_summary.csv")



saved eda table to medlineplus_eda_summary.csv
